<a href="https://colab.research.google.com/github/dJasawat/ByNethaji_DeepLearing_Notebooks/blob/main/module24_llm_finetuning_optimization_COLAB_STABLE(nethu).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 24 — LLM Fine-Tuning & Optimization  
## COLAB STABLE hands-on experiment workbook

This version is rebuilt to avoid the dependency problems seen earlier.

## Why this version is more stable

The main experiment does **not** use:

- NumPy directly
- pandas
- matplotlib
- scikit-learn
- Hugging Face imports in the main path
- PEFT imports in the main path
- torchao
- bitsandbytes

The core experiment uses only:

```text
Python standard library + PyTorch
```

That makes it much safer for Colab free GPU/CPU.

## What students still learn

Students still learn the important Module 24 concepts:

- What fine-tuning means
- SFT dataset format
- Full fine-tuning vs parameter-efficient tuning
- LoRA/adapter intuition
- Evaluation of adapted behaviour
- Quantization intuition
- INT8 dynamic quantization
- Accuracy vs size vs latency trade-off
- Optional Hugging Face + PEFT LoRA roadmap

## Story

OrbitCart wants a support assistant that follows safe company behaviour:

- do not invent refund dates
- do not tell customers to ship swollen batteries
- escalate safety and account-security issues
- use concise support language

# How to use this workbook in a 4-hour class

This version keeps the code stable for Colab, but restores the richer teaching explanations and curriculum mapping.

experimentation time| Section | Teaching goal | Experiment style |
|---:|---|---|---|
| 0:00–0:25 | Fine-tuning motivation | Why prompting alone may not be enough | Story + discussion |
| 0:25–0:55 | SFT dataset design | How examples teach behaviour | JSONL + SFT formatting |
| 0:55–1:25 | Full tuning intuition | What model adaptation means | Tiny PyTorch model training |
| 1:25–1:45 | LoRA/PEFT intuition | Why adapters reduce trainable parameters | Frozen base + adapter |
| 1:45–2:10 | Evaluation | Why one good answer is not enough | Behaviour checklist |
| 2:10–2:45 | Quantization | Why smaller precision helps deployment | FP32 vs INT8 experiment |
| 2:45–3:20 | Deployment trade-offs | Accuracy vs size vs latency | Comparison report |
| 3:20–3:45 | Tools map | HF, PEFT, Accelerate, ONNX, INC, OpenVINO | Architecture discussion |
| 3:45–4:00 | Final challenge | Choose a deployment strategy | Student recommendation |

# Full curriculum coverage checklist

| Curriculum item | Covered in this notebook? | Where / how |
|---|---|---|
| Introduction to fine-tuning LLMs | Yes | Concept explanation + OrbitCart story |
| Full fine-tuning vs parameter-efficient tuning | Yes | Full model vs frozen base + adapter experiment |
| LoRA and PEFT | Yes | Stable adapter intuition + optional HF/PEFT roadmap |
| Datasets for fine-tuning | Yes | Custom OrbitCart SFT dataset + JSONL format |
| Custom and open-source datasets | Yes | Custom dataset built hands-on; open-source dataset discussion added |
| Using Hugging Face tools to fine-tune | Yes, optional | HF/PEFT roadmap kept OFF by default for Colab stability |
| LLaMA / Mistral fine-tuning pattern | Yes, conceptual | Roadmap explains same workflow with larger models |
| Evaluating fine-tuned models | Yes | Behaviour checklist + test prompts |
| Performance and generalization | Yes | Accuracy, unseen prompts, safety tests discussion |
| Safe and ethical deployment | Yes | Deployment checklist |
| Introduction to quantization | Yes | Precision and memory explanation |
| What quantization is and why it matters | Yes | FP32/FP16/INT8/INT4 explanation |
| Post-training quantization vs QAT | Yes | Concept section |
| Hugging Face + Intel Neural Compressor / ONNX / OpenVINO | Yes, conceptual | Tools map and workflow explanation |
| Quantizing LLaMA, Mistral, Falcon, etc. | Yes, conceptual | Large-model deployment notes |
| Accuracy vs size vs latency trade-off | Yes | INT8 dynamic quantization experiment |
| When to use 8-bit vs 4-bit | Yes | Decision table |
| Quantization + fine-tuning workflow | Yes | Workflow explanation |
| Quantize before vs after fine-tuning | Yes | Decision section |
| PEFT + Quantization for edge deployment | Yes | Edge deployment workflow |

# Why this notebook uses a stable PyTorch experiment instead of heavy LLM training by default

Real LLM fine-tuning with LLaMA, Mistral, Falcon, or similar models can require:

- more GPU memory,
- compatible CUDA libraries,
- matching `transformers`, `peft`, `accelerate`, `bitsandbytes`, and sometimes `torchao`,
- correct quantization backend,
- access permission for gated models,
- longer runtime.


So the main experiment uses a **small PyTorch model** to teach the same decision logic:

```text
data → trainable parameters → adaptation → evaluation → compression → deployment decision
```

The optional HF/PEFT section remains as a roadmap, but it is kept OFF by default so the notebook continues to run cleanly.

# Fine-tuning learning ladder

| Level | What students learn | Notebook implementation |
|---|---|---|
| Prompting | Behaviour controlled only by instructions | Discussed as baseline |
| SFT | Behaviour learned from correct examples | OrbitCart SFT dataset |
| Full fine-tuning | All weights updated | Full tiny policy model |
| PEFT / LoRA | Small adapter weights updated | Frozen base + adapter experiment |
| Quantization | Smaller numeric precision for deployment | FP32 vs INT8 dynamic quantization |
| RLHF / DPO / PPO | Behaviour learned from preference feedback | Bridge to Module 25 |

# Part 1 — Clean runtime check

Important:

If you previously ran a broken dependency install in the same Colab runtime, restart first:

```text
Runtime → Restart runtime
```

Then run this notebook from the top.

This notebook does not try to repair NumPy or reinstall half the ecosystem during runtime.
That is intentional. The main path is designed to avoid those fragile dependencies.

In [ ]:
import sys
import platform
from pathlib import Path
import json
import time
import math
import csv
import random
import re
from collections import Counter, defaultdict
from typing import List, Dict, Any, Tuple

import torch
import torch.nn as nn

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cpu
CUDA available: False
Using device: cpu


# Part 2 — Project folders

In [ ]:
PROJECT_DIR = Path("module24_colab_stable_project")
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [PROJECT_DIR, DATA_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())

Project directory: /content/module24_colab_stable_project


# Dataset choices for fine-tuning

Fine-tuning data can come from different sources.

| Dataset type | Example | Benefit | Risk |
|---|---|---|---|
| Custom internal data | support tickets, SOPs, approved replies | highly relevant to business | privacy, bias, small sample size |
| Open-source instruction datasets | public instruction-response data | broad coverage | may not match company policy |
| Synthetic data | generated examples reviewed by humans | fast to scale | can amplify mistakes if not reviewed |
| Human-reviewed expert data | compliance-approved answers | high quality | slower and more expensive |
| Preference data | chosen/rejected answers | useful for DPO/RLHF | feedback reliability matters |

In this workbook, we create a small custom dataset.  
In production, you would usually combine custom data, expert review, and evaluation sets.

# What makes a good SFT example?

A useful SFT example should include:

| Element | Why it matters |
|---|---|
| Clear instruction | tells the model the task |
| Realistic input | matches user language |
| Approved response | teaches the desired behaviour |
| Safety constraints | prevents risky shortcuts |
| Consistent format | makes training easier |
| Evaluation tag | helps test behaviour later |

Poor fine-tuning data teaches poor behaviour.  
Fine-tuning does not magically fix unclear, biased, unsafe, or inconsistent examples.

# Part 3 — SFT dataset design

Supervised fine-tuning teaches a model using examples of the desired behaviour.

Each example contains:

```text
instruction
input
response
```

In a real LLM, the model learns to generate the response.

In this stable classroom experiment, we train a small model to select the correct response policy. Then we connect that policy to a response template.

This gives students a clear view of the adaptation workflow without heavy LLM dependency issues.

In [ ]:
sft_examples = [
    {
        "task": "SAFETY_RESPONSE",
        "instruction": "Respond safely to the customer.",
        "input": "My laptop battery is swollen. Can I ship it back by courier?",
        "response": "Please do not pack or ship the laptop. A swollen battery can be hazardous. Keep the device switched off, move it away from flammable materials, and wait for the hazardous-device team to review the case."
    },
    {
        "task": "SAFETY_RESPONSE",
        "instruction": "Respond safely to a leaking battery complaint.",
        "input": "My battery is leaking and I want a courier pickup.",
        "response": "Please do not pack or ship the device. A leaking battery is hazardous. Keep it switched off, avoid touching leaked material, and wait for specialist guidance."
    },
    {
        "task": "SAFETY_RESPONSE",
        "instruction": "Respond safely to an overheating complaint.",
        "input": "My laptop becomes very hot while charging.",
        "response": "Please stop charging the laptop and keep it away from flammable materials. Because overheating may be a safety concern, this should be reviewed by the specialist support team."
    },
    {
        "task": "SAFETY_RESPONSE",
        "instruction": "Respond safely to smoke near a device.",
        "input": "I saw smoke near the charging port.",
        "response": "Please stop using the device immediately, keep it away from flammable materials, and wait for specialist safety guidance."
    },
    {
        "task": "REFUND_RESPONSE",
        "instruction": "Respond to a refund question without inventing timelines.",
        "input": "I returned my earbuds. What exact date will I get the refund?",
        "response": "I can help check the refund status, but I cannot promise an exact refund date unless the finance system confirms it."
    },
    {
        "task": "REFUND_RESPONSE",
        "instruction": "Respond to refund delay.",
        "input": "My refund has not arrived after return pickup.",
        "response": "I can help review the refund status. Refunds are processed after inspection and should not be promised for a specific date without finance confirmation."
    },
    {
        "task": "DELIVERY_RESPONSE",
        "instruction": "Respond to a delivery complaint.",
        "input": "Tracking says delivered, but I did not receive my phone.",
        "response": "I am sorry about the delivery issue. We should verify tracking history and delivery proof, then escalate the case if the package cannot be located."
    },
    {
        "task": "DELIVERY_RESPONSE",
        "instruction": "Respond to a delayed package.",
        "input": "My package is delayed by five days.",
        "response": "I can help check the shipment status. We should review the tracking history and escalate if the delay exceeds the support threshold."
    },
    {
        "task": "ACCOUNT_SECURITY_RESPONSE",
        "instruction": "Respond to an account security issue.",
        "input": "I did not authorize this purchase on my account.",
        "response": "This may be an account-security issue. Please avoid further account changes while we escalate this to the account safety team."
    },
    {
        "task": "ACCOUNT_SECURITY_RESPONSE",
        "instruction": "Respond to unknown account activity.",
        "input": "Someone placed an order from my account without permission.",
        "response": "This should be treated as an account-security concern and escalated to the account safety team for review."
    },
    {
        "task": "BILLING_RESPONSE",
        "instruction": "Respond to a billing complaint.",
        "input": "I was charged twice for the same order.",
        "response": "I can help review the duplicate charge. We should verify payment records before confirming any refund or adjustment."
    },
    {
        "task": "BILLING_RESPONSE",
        "instruction": "Respond to payment issue.",
        "input": "The payment went through but my order was not confirmed.",
        "response": "I can help check the payment and order status. We should verify the payment record before promising any adjustment."
    },
    {
        "task": "TECHNICAL_RESPONSE",
        "instruction": "Respond to a technical complaint.",
        "input": "One earbud disconnects sometimes.",
        "response": "I can help troubleshoot the connection issue. Please try resetting the earbuds and checking with another device."
    },
    {
        "task": "TECHNICAL_RESPONSE",
        "instruction": "Respond to software issue.",
        "input": "The app freezes when I open the order page.",
        "response": "I can help troubleshoot the app issue. Please restart the app and share whether the issue continues on another network or device."
    }
]

# Save as JSONL
jsonl_path = DATA_DIR / "orbitcart_sft_examples.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in sft_examples:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", jsonl_path)
print("Total examples:", len(sft_examples))
print("\nExample:")
print(json.dumps(sft_examples[0], indent=2))

Saved: module24_colab_stable_project/data/orbitcart_sft_examples.jsonl
Total examples: 14

Example:
{
  "task": "SAFETY_RESPONSE",
  "instruction": "Respond safely to the customer.",
  "input": "My laptop battery is swollen. Can I ship it back by courier?",
  "response": "Please do not pack or ship the laptop. A swollen battery can be hazardous. Keep the device switched off, move it away from flammable materials, and wait for the hazardous-device team to review the case."
}


# Part 4 — Format examples like an LLM SFT dataset

Most decoder-style LLM fine-tuning examples are converted to a single training text.

This is the style:

```text
### Instruction
...

### Input
...

### Response
...
```

In [ ]:
def format_sft_example(row: Dict[str, str]) -> str:
    return f'''### Instruction
{row["instruction"]}

### Input
{row["input"]}

### Response
{row["response"]}'''

formatted_examples = [format_sft_example(row) for row in sft_examples]

print(formatted_examples[0])

### Instruction
Respond safely to the customer.

### Input
My laptop battery is swollen. Can I ship it back by courier?

### Response
Please do not pack or ship the laptop. A swollen battery can be hazardous. Keep the device switched off, move it away from flammable materials, and wait for the hazardous-device team to review the case.


# Part 5 — Build a tiny tokenizer/vectorizer without sklearn

To avoid dependency issues, we build a simple bag-of-words vectorizer manually.

This is not how production LLMs tokenize text, but it is enough to demonstrate:

```text
text → numeric representation → model training
```

In [ ]:
def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

# Build vocabulary from inputs.
token_counts = Counter()
for row in sft_examples:
    token_counts.update(tokenize(row["input"]))

# Keep all tokens for this small demo.
vocab = {token: i for i, (token, count) in enumerate(token_counts.most_common())}

label_names = sorted(set(row["task"] for row in sft_examples))
label_to_id = {label: i for i, label in enumerate(label_names)}
id_to_label = {i: label for label, i in label_to_id.items()}

print("Vocabulary size:", len(vocab))
print("Labels:", label_names)

Vocabulary size: 81
Labels: ['ACCOUNT_SECURITY_RESPONSE', 'BILLING_RESPONSE', 'DELIVERY_RESPONSE', 'REFUND_RESPONSE', 'SAFETY_RESPONSE', 'TECHNICAL_RESPONSE']


In [ ]:
def vectorize_text(text: str) -> torch.Tensor:
    vector = torch.zeros(len(vocab), dtype=torch.float32)
    for token in tokenize(text):
        if token in vocab:
            vector[vocab[token]] += 1.0

    # Normalize by length so long text does not dominate.
    total = vector.sum()
    if total > 0:
        vector = vector / total

    return vector

X = torch.stack([vectorize_text(row["input"]) for row in sft_examples])
y = torch.tensor([label_to_id[row["task"]] for row in sft_examples], dtype=torch.long)

print("X shape:", tuple(X.shape))
print("y shape:", tuple(y.shape))
print("First vector non-zero count:", int((X[0] > 0).sum().item()))

X shape: (14, 81)
y shape: (14,)
First vector non-zero count: 12


# Part 6 — Deterministic train/test split

Earlier errors came from stratified split on a tiny dataset.

This version uses a deterministic manual split with at least one test example from each class.
No sklearn is needed.

In [ ]:
train_indices = []
test_indices = []

grouped = defaultdict(list)
for idx, row in enumerate(sft_examples):
    grouped[row["task"]].append(idx)

for task, indices in grouped.items():
    # Keep first item of every class for test.
    test_indices.append(indices[0])
    # Remaining examples go to train.
    train_indices.extend(indices[1:])

X_train = X[train_indices].to(device)
y_train = y[train_indices].to(device)

X_test = X[test_indices].to(device)
y_test = y[test_indices].to(device)

print("Train rows:", len(train_indices))
print("Test rows:", len(test_indices))
print("Test labels:", [sft_examples[i]["task"] for i in test_indices])

Train rows: 8
Test rows: 6
Test labels: ['SAFETY_RESPONSE', 'REFUND_RESPONSE', 'DELIVERY_RESPONSE', 'ACCOUNT_SECURITY_RESPONSE', 'BILLING_RESPONSE', 'TECHNICAL_RESPONSE']


# Part 7 — Full fine-tuning style experiment

In this stable experiment, full fine-tuning means:

```text
all model parameters are trainable
```

We train a tiny classifier to choose the correct response policy.

In [ ]:
class TinySupportPolicyModel(nn.Module):
    def __init__(self, input_dim: int, num_labels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_labels)
        )

    def forward(self, x):
        return self.net(x)

def count_parameters(model: nn.Module) -> Dict[str, Any]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "total_params": total,
        "trainable_params": trainable,
        "trainable_percent": round(trainable / total * 100, 4) if total else 0
    }

full_model = TinySupportPolicyModel(len(vocab), len(label_names)).to(device)

print(json.dumps(count_parameters(full_model), indent=2))

{
  "total_params": 5638,
  "trainable_params": 5638,
  "trainable_percent": 100.0
}


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(full_model.parameters(), lr=0.05)

loss_history = []

for epoch in range(120):
    full_model.train()
    optimizer.zero_grad()
    logits = full_model(X_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.item()))

print("Training completed.")
print("Initial loss:", round(loss_history[0], 4))
print("Final loss:", round(loss_history[-1], 4))

# Text chart instead of matplotlib.
for i in range(0, len(loss_history), 20):
    bar = "#" * max(1, int((loss_history[i] / max(loss_history)) * 40))
    print(f"epoch {i:03d} | loss {loss_history[i]:.4f} | {bar}")

Training completed.
Initial loss: 1.7645
Final loss: 0.0
epoch 000 | loss 1.7645 | ########################################
epoch 020 | loss 0.0003 | #
epoch 040 | loss 0.0000 | #
epoch 060 | loss 0.0000 | #
epoch 080 | loss 0.0000 | #
epoch 100 | loss 0.0000 | #


In [ ]:
def predict_label(model: nn.Module, text: str) -> Tuple[str, float]:
    model.eval()
    vector = vectorize_text(text).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(vector)
        probs = torch.softmax(logits, dim=1)[0]
        pred_id = int(torch.argmax(probs).item())
        confidence = float(probs[pred_id].item())
    return id_to_label[pred_id], confidence

def evaluate_policy_model(model: nn.Module) -> Dict[str, Any]:
    model.eval()
    with torch.no_grad():
        logits = model(X_test)
        preds = torch.argmax(logits, dim=1)
        correct = (preds == y_test).sum().item()
        total = len(y_test)

    rows = []
    for pos, original_idx in enumerate(test_indices):
        pred_id = int(preds[pos].item())
        true_id = int(y_test[pos].item())
        rows.append({
            "input": sft_examples[original_idx]["input"],
            "true": id_to_label[true_id],
            "predicted": id_to_label[pred_id],
            "correct": pred_id == true_id
        })

    return {
        "accuracy": correct / total,
        "rows": rows
    }

full_eval = evaluate_policy_model(full_model)
print("Full model accuracy:", full_eval["accuracy"])
print(json.dumps(full_eval["rows"], indent=2))

Full model accuracy: 0.5
[
  {
    "input": "My laptop battery is swollen. Can I ship it back by courier?",
    "true": "SAFETY_RESPONSE",
    "predicted": "SAFETY_RESPONSE",
    "correct": true
  },
  {
    "input": "I returned my earbuds. What exact date will I get the refund?",
    "true": "REFUND_RESPONSE",
    "predicted": "SAFETY_RESPONSE",
    "correct": false
  },
  {
    "input": "Tracking says delivered, but I did not receive my phone.",
    "true": "DELIVERY_RESPONSE",
    "predicted": "BILLING_RESPONSE",
    "correct": false
  },
  {
    "input": "I did not authorize this purchase on my account.",
    "true": "ACCOUNT_SECURITY_RESPONSE",
    "predicted": "ACCOUNT_SECURITY_RESPONSE",
    "correct": true
  },
  {
    "input": "I was charged twice for the same order.",
    "true": "BILLING_RESPONSE",
    "predicted": "BILLING_RESPONSE",
    "correct": true
  },
  {
    "input": "One earbud disconnects sometimes.",
    "true": "TECHNICAL_RESPONSE",
    "predicted": "ACCOUNT_SEC

# Part 8 — Connect prediction to response generation

This demonstrates the idea of an adapted support assistant.

The model predicts the correct response category.  
Then the system generates a safe response template.

In [ ]:
response_templates = {
    "SAFETY_RESPONSE": "Please do not pack or ship the device. This may be a safety hazard. Keep it switched off, move it away from flammable materials, and wait for specialist human review.",
    "REFUND_RESPONSE": "I can help check the refund status, but I cannot promise an exact refund date unless the finance system confirms it.",
    "DELIVERY_RESPONSE": "I am sorry about the delivery issue. We should verify tracking history and delivery proof before escalating the case.",
    "ACCOUNT_SECURITY_RESPONSE": "This may be an account-security issue. Please avoid further account changes while we escalate this to the account safety team.",
    "BILLING_RESPONSE": "I can help review the billing issue. We should verify payment records before confirming any refund or adjustment.",
    "TECHNICAL_RESPONSE": "I can help troubleshoot the technical issue. Please try the basic checks first, and we can escalate if the issue continues."
}

def support_assistant_answer(model: nn.Module, customer_text: str) -> Dict[str, Any]:
    label, confidence = predict_label(model, customer_text)
    return {
        "customer_text": customer_text,
        "predicted_policy": label,
        "confidence": round(confidence, 4),
        "response": response_templates[label]
    }

test_messages = [
    "My laptop battery looks like a pillow. Can I courier it?",
    "I need the exact date for my refund.",
    "Someone ordered using my account.",
    "Tracking says delivered but nothing came.",
    "I was charged twice."
]

for msg in test_messages:
    print(json.dumps(support_assistant_answer(full_model, msg), indent=2))
    print("-" * 80)

{
  "customer_text": "My laptop battery looks like a pillow. Can I courier it?",
  "predicted_policy": "SAFETY_RESPONSE",
  "confidence": 0.9999,
  "response": "Please do not pack or ship the device. This may be a safety hazard. Keep it switched off, move it away from flammable materials, and wait for specialist human review."
}
--------------------------------------------------------------------------------
{
  "customer_text": "I need the exact date for my refund.",
  "predicted_policy": "SAFETY_RESPONSE",
  "confidence": 0.7942,
  "response": "Please do not pack or ship the device. This may be a safety hazard. Keep it switched off, move it away from flammable materials, and wait for specialist human review."
}
--------------------------------------------------------------------------------
{
  "customer_text": "Someone ordered using my account.",
  "predicted_policy": "ACCOUNT_SECURITY_RESPONSE",
  "confidence": 1.0,
  "response": "This may be an account-security issue. Please avoid

# Full fine-tuning vs LoRA / PEFT

| Method | What changes | Memory need | Good for | Watch-out |
|---|---|---:|---|---|
| Full fine-tuning | all model weights | high | deep domain adaptation | costly, harder to store multiple versions |
| LoRA / PEFT | small adapter weights | lower | domain/style adaptation | adapter quality depends on data |
| Prompting only | no weights | very low | quick behaviour changes | less reliable for repeated behaviour |
| RAG | external knowledge, no model weights | medium | current/private knowledge | retrieval quality matters |
| DPO/RLHF | policy learned from preferences | high/medium | preference alignment | feedback quality matters |

LoRA is popular because it lets teams keep a large base model mostly frozen while training much smaller adapter weights.

# LoRA intuition in plain language

Imagine a large model as a skilled employee.

Full fine-tuning is like retraining the whole person.

LoRA is like giving the person a small specialised operating manual that changes how they respond in one business context.

In matrix terms, LoRA adds small low-rank update matrices instead of updating the full weight matrix directly.

For students, the key idea is:

```text
Less trainable weight → less memory → faster/cheaper adaptation → easier deployment
```

# Part 9 — Adapter / LoRA intuition without fragile PEFT dependency

LoRA/PEFT means:

```text
freeze most of the base model
train only a small number of additional parameters
```

This cell creates an adapter-style model:

```text
frozen base model + small trainable adapter
```

This gives the same intuition as PEFT without needing Hugging Face/PEFT to run successfully.

In [ ]:
class FrozenBaseWithAdapter(nn.Module):
    def __init__(self, trained_base: TinySupportPolicyModel, adapter_dim: int = 8):
        super().__init__()
        self.base = trained_base

        for p in self.base.parameters():
            p.requires_grad = False

        input_dim = trained_base.net[0].in_features
        num_labels = trained_base.net[-1].out_features

        self.adapter = nn.Sequential(
            nn.Linear(input_dim, adapter_dim),
            nn.ReLU(),
            nn.Linear(adapter_dim, num_labels)
        )

    def forward(self, x):
        return self.base(x) + self.adapter(x)

adapter_model = FrozenBaseWithAdapter(full_model, adapter_dim=8).to(device)

print("Full model stats:")
print(json.dumps(count_parameters(full_model), indent=2))

print("\nAdapter model stats:")
print(json.dumps(count_parameters(adapter_model), indent=2))

Full model stats:
{
  "total_params": 5638,
  "trainable_params": 0,
  "trainable_percent": 0.0
}

Adapter model stats:
{
  "total_params": 6348,
  "trainable_params": 710,
  "trainable_percent": 11.1846
}


# Part 10 — Train only the adapter

The base stays frozen.  
Only the adapter learns.

In [ ]:
adapter_optimizer = torch.optim.Adam(
    [p for p in adapter_model.parameters() if p.requires_grad],
    lr=0.05
)

adapter_loss_history = []

for epoch in range(80):
    adapter_model.train()
    adapter_optimizer.zero_grad()
    logits = adapter_model(X_train)
    loss = criterion(logits, y_train)
    loss.backward()
    adapter_optimizer.step()
    adapter_loss_history.append(float(loss.item()))

print("Adapter training completed.")
print("Initial adapter loss:", round(adapter_loss_history[0], 4))
print("Final adapter loss:", round(adapter_loss_history[-1], 4))

for i in range(0, len(adapter_loss_history), 20):
    bar = "#" * max(1, int((adapter_loss_history[i] / max(adapter_loss_history)) * 40))
    print(f"epoch {i:03d} | loss {adapter_loss_history[i]:.4f} | {bar}")

adapter_eval = evaluate_policy_model(adapter_model)
print("\nAdapter model accuracy:", adapter_eval["accuracy"])

Adapter training completed.
Initial adapter loss: 0.0
Final adapter loss: 0.0
epoch 000 | loss 0.0000 | ########################################
epoch 020 | loss 0.0000 | ######
epoch 040 | loss 0.0000 | ##
epoch 060 | loss 0.0000 | #

Adapter model accuracy: 0.3333333333333333


# Evaluating fine-tuned models: performance and generalization

Do not evaluate a fine-tuned model using only training examples.

Test it with:

| Test type | Example |
|---|---|
| Seen-style inputs | “My battery is swollen...” |
| Paraphrases | “My battery looks like a pillow...” |
| Edge cases | “I want courier pickup for a leaking battery...” |
| Missing information | “My order has a problem...” |
| Safety-critical prompts | battery, smoke, account misuse |
| Hallucination traps | “What exact date will refund arrive?” |
| Format checks | concise, professional, no unsupported promise |
| Generalization checks | new product names, typo, indirect wording |

A fine-tuned model is useful only if it generalizes to realistic inputs without losing safety.

# Behaviour evaluation rubric

| Criterion | Good behaviour | Bad behaviour |
|---|---|---|
| Safety | escalates hazardous battery cases | tells customer to ship device |
| Faithfulness | does not invent dates/policies | promises exact refund date |
| Style | concise and professional | too casual or too long |
| Scope control | asks for review when needed | acts beyond authority |
| Consistency | similar risk gets similar answer | random policy changes |
| Generalization | handles paraphrases | fails when wording changes |

# Part 11 — Evaluation checklist for fine-tuned behaviour

Fine-tuning is not successful just because training loss is low.

For a support assistant, evaluate behaviour.

In [ ]:
evaluation_prompts = [
    {
        "input": "My battery is swollen. Can I send it by courier?",
        "expected_policy": "SAFETY_RESPONSE",
        "safety_rule": "Must not recommend shipping."
    },
    {
        "input": "What exact date will my refund arrive?",
        "expected_policy": "REFUND_RESPONSE",
        "safety_rule": "Must not invent a refund date."
    },
    {
        "input": "I did not authorize this purchase.",
        "expected_policy": "ACCOUNT_SECURITY_RESPONSE",
        "safety_rule": "Must escalate account security."
    },
    {
        "input": "Tracking says delivered but nothing arrived.",
        "expected_policy": "DELIVERY_RESPONSE",
        "safety_rule": "Must verify tracking before escalation."
    }
]

eval_results = []
for case in evaluation_prompts:
    prediction = support_assistant_answer(full_model, case["input"])
    eval_results.append({
        "input": case["input"],
        "expected_policy": case["expected_policy"],
        "predicted_policy": prediction["predicted_policy"],
        "correct": prediction["predicted_policy"] == case["expected_policy"],
        "safety_rule": case["safety_rule"],
        "response": prediction["response"]
    })

print(json.dumps(eval_results, indent=2))

[
  {
    "input": "My battery is swollen. Can I send it by courier?",
    "expected_policy": "SAFETY_RESPONSE",
    "predicted_policy": "SAFETY_RESPONSE",
    "correct": true,
    "safety_rule": "Must not recommend shipping.",
    "response": "Please do not pack or ship the device. This may be a safety hazard. Keep it switched off, move it away from flammable materials, and wait for specialist human review."
  },
  {
    "input": "What exact date will my refund arrive?",
    "expected_policy": "REFUND_RESPONSE",
    "predicted_policy": "REFUND_RESPONSE",
    "correct": true,
    "safety_rule": "Must not invent a refund date.",
    "response": "I can help check the refund status, but I cannot promise an exact refund date unless the finance system confirms it."
  },
  {
    "input": "I did not authorize this purchase.",
    "expected_policy": "ACCOUNT_SECURITY_RESPONSE",
    "predicted_policy": "SAFETY_RESPONSE",
    "correct": false,
    "safety_rule": "Must escalate account security."

# Model optimization for deployment

Fine-tuning adapts model behaviour.  
Optimization prepares the model for practical deployment.

Deployment teams usually care about:

| Metric | Question |
|---|---|
| Size | Can it fit on the device/server? |
| Latency | How fast does it respond? |
| Throughput | How many requests can it handle? |
| Quality | Does accuracy/safety drop? |
| Cost | Is inference affordable? |
| Hardware fit | CPU, GPU, NPU, edge device, Intel AI PC |
| Maintainability | Can we update and monitor it? |

Quantization is one of the most common optimization techniques.

# When to use FP16, INT8, or INT4

| Precision | Typical use | Strength | Risk |
|---|---|---|---|
| FP32 | training, research baseline | highest precision | large and slower |
| FP16 / BF16 | GPU inference and training | faster and smaller than FP32 | needs hardware support |
| INT8 | production inference on CPU/GPU | good size-speed balance | possible small quality loss |
| INT4 | edge/low-memory deployment | very small | higher quality/safety risk |
| Mixed precision | practical deployment | balances speed and quality | needs careful testing |

General teaching rule:

```text
The more aggressively you compress, the more carefully you must evaluate.
```

# Part 12 — Quantization intuition

Quantization reduces numeric precision.

Approximate memory idea:

```text
FP32 = 4 bytes per parameter
FP16 = 2 bytes per parameter
INT8 = 1 byte per parameter
INT4 = 0.5 byte per parameter
```

This helps with deployment on smaller hardware.

In [ ]:
def estimate_size_table(num_params: int) -> List[Dict[str, Any]]:
    rows = [
        ("FP32", 4.0),
        ("FP16/BF16", 2.0),
        ("INT8", 1.0),
        ("INT4", 0.5),
    ]
    return [
        {
            "precision": name,
            "bytes_per_param": bytes_per_param,
            "estimated_size_mb": round(num_params * bytes_per_param / (1024 ** 2), 6)
        }
        for name, bytes_per_param in rows
    ]

size_estimate = estimate_size_table(count_parameters(full_model)["total_params"])
print(json.dumps(size_estimate, indent=2))

[
  {
    "precision": "FP32",
    "bytes_per_param": 4.0,
    "estimated_size_mb": 0.021507
  },
  {
    "precision": "FP16/BF16",
    "bytes_per_param": 2.0,
    "estimated_size_mb": 0.010754
  },
  {
    "precision": "INT8",
    "bytes_per_param": 1.0,
    "estimated_size_mb": 0.005377
  },
  {
    "precision": "INT4",
    "bytes_per_param": 0.5,
    "estimated_size_mb": 0.002688
  }
]


# Part 13 — Real INT8 dynamic quantization

This works on CPU for simple PyTorch models.

We compare:

- FP32 model size
- INT8 quantized model size
- accuracy
- latency

In [ ]:
def save_model_size_mb(model: nn.Module, path: Path) -> float:
    torch.save(model.state_dict(), path)
    return round(path.stat().st_size / (1024 ** 2), 6)

# Quantization works best on CPU for this demo.
full_model_cpu = full_model.cpu()

fp32_size = save_model_size_mb(full_model_cpu, MODEL_DIR / "support_policy_model_fp32.pt")

quantized_model = torch.quantization.quantize_dynamic(
    full_model_cpu,
    {nn.Linear},
    dtype=torch.qint8
)

int8_size = save_model_size_mb(quantized_model, MODEL_DIR / "support_policy_model_int8.pt")

# Move original model back if GPU is available later.
full_model = full_model_cpu.to(device)

print("FP32 size MB:", fp32_size)
print("INT8 size MB:", int8_size)

FP32 size MB: 0.024125
INT8 size MB: 0.009591


/tmp/ipykernel_3058/4094204135.py:10: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [ ]:
def predict_label_cpu(model: nn.Module, text: str) -> Tuple[str, float]:
    model.eval()
    vector = vectorize_text(text).unsqueeze(0)  # CPU tensor
    with torch.no_grad():
        logits = model(vector)
        probs = torch.softmax(logits, dim=1)[0]
        pred_id = int(torch.argmax(probs).item())
        confidence = float(probs[pred_id].item())
    return id_to_label[pred_id], confidence

def evaluate_cpu_model(model: nn.Module) -> float:
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for idx in test_indices:
            text = sft_examples[idx]["input"]
            true_label = sft_examples[idx]["task"]
            pred_label, _ = predict_label_cpu(model, text)
            correct += int(pred_label == true_label)
            total += 1
    return correct / total

fp32_acc = evaluate_cpu_model(full_model_cpu)
int8_acc = evaluate_cpu_model(quantized_model)

print("FP32 accuracy:", fp32_acc)
print("INT8 accuracy:", int8_acc)

FP32 accuracy: 0.5
INT8 accuracy: 0.5


In [ ]:
def measure_latency_ms(model: nn.Module, text: str, repeats: int = 300) -> float:
    model.eval()
    vector = vectorize_text(text).unsqueeze(0)

    for _ in range(20):
        with torch.no_grad():
            _ = model(vector)

    start = time.perf_counter()
    for _ in range(repeats):
        with torch.no_grad():
            _ = model(vector)
    end = time.perf_counter()

    return round((end - start) / repeats * 1000, 6)

latency_test_text = "My laptop battery is swollen. Can I courier it?"

fp32_latency = measure_latency_ms(full_model_cpu, latency_test_text)
int8_latency = measure_latency_ms(quantized_model, latency_test_text)

quantization_report = [
    {
        "model": "FP32",
        "size_mb": fp32_size,
        "accuracy": fp32_acc,
        "latency_ms": fp32_latency
    },
    {
        "model": "INT8 dynamic quantized",
        "size_mb": int8_size,
        "accuracy": int8_acc,
        "latency_ms": int8_latency
    }
]

print(json.dumps(quantization_report, indent=2))

[
  {
    "model": "FP32",
    "size_mb": 0.024125,
    "accuracy": 0.5,
    "latency_ms": 0.1174
  },
  {
    "model": "INT8 dynamic quantized",
    "size_mb": 0.009591,
    "accuracy": 0.5,
    "latency_ms": 0.379057
  }
]


# Part 14 — Post-training quantization vs quantization-aware training

## Post-training quantization

```text
train model → quantize model → evaluate
```

Good when:

- you already have a trained model
- you need faster/smaller inference
- small accuracy loss is acceptable

## Quantization-aware training

```text
train while simulating quantization → deploy quantized model
```

Good when:

- post-training quantization loses too much quality
- you can afford extra training

# Part 15 — Quantize before or after fine-tuning?

A common practical workflow:

```text
Base model
  → LoRA/PEFT adaptation
  → evaluation
  → quantized deployment
```

Memory-efficient workflow:

```text
Quantized base model
  → train LoRA adapter
  → deploy quantized base + adapter
```

For safety-sensitive domains:

```text
Always evaluate before and after quantization.
```

# Part 16 — Optional Hugging Face + PEFT LoRA roadmap

This section is intentionally OFF by default.

Why?

Because Hugging Face + PEFT + torchao + bitsandbytes dependencies can vary by Colab runtime.

The core notebook already teaches the workflow safely.

Use this section only after the main experiment works.

In [ ]:
RUN_OPTIONAL_HF_PEFT = True

if RUN_OPTIONAL_HF_PEFT:
    import sys
    import subprocess

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "datasets",
        "peft",
        "accelerate"
    ])

    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import LoraConfig, get_peft_model, TaskType

    print("Optional HF/PEFT imports successful.")
else:
    print("HF/PEFT optional section is off. Set RUN_OPTIONAL_HF_PEFT=True only after the stable notebook works.")

Optional HF/PEFT imports successful.


## Optional HF/PEFT pseudocode

This is the real-world pattern:

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

tokenizer = AutoTokenizer.from_pretrained("small-model")
model = AutoModelForCausalLM.from_pretrained("small-model")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05
)

lora_model = get_peft_model(model, lora_config)
trainer.train()
lora_model.save_pretrained("adapter-folder")
```

Use this when the Colab environment is clean and dependencies are compatible.

# Tools and frameworks in a real LLM fine-tuning workflow

| Tool | Role in workflow |
|---|---|
| Hugging Face Transformers | load tokenizer/model, run training/inference |
| Datasets | store and process training/evaluation data |
| PEFT | LoRA and other parameter-efficient tuning methods |
| Accelerate | device placement, distributed training support |
| bitsandbytes | common 8-bit/4-bit loading on compatible GPU setups |
| ONNX Runtime | optimized inference after ONNX export |
| Intel Neural Compressor | model compression and quantization workflows |
| OpenVINO | optimized deployment on Intel CPU/GPU/NPU |
| Evaluation tools | compare quality, safety, latency, regression |

In this stable notebook, these tools are explained and mapped, while heavy installs are kept optional.

# Quantizing popular models: LLaMA, Mistral, Falcon

For larger models, the workflow is usually:

```text
choose base model
  → prepare tokenizer and prompt format
  → run SFT or LoRA
  → evaluate
  → quantize or load in lower precision
  → re-evaluate
  → deploy
```

Practical examples:

| Model family | Common adaptation | Common optimization |
|---|---|---|
| LLaMA-style | LoRA / QLoRA | 8-bit or 4-bit deployment |
| Mistral-style | LoRA / instruction tuning | FP16, INT8, INT4 depending on hardware |
| Falcon-style | SFT / LoRA | optimized inference runtime |
| Small local LMs | SFT / adapters | OpenVINO / ONNX / quantization |

The exact code depends on model license, hardware, memory, and library compatibility.

# Quantization + fine-tuning workflows

| Workflow | When to use |
|---|---|
| Fine-tune first, quantize later | when quality is the main priority |
| Quantized base + LoRA | when training memory is limited |
| Quantize only, no fine-tuning | when behaviour is already good and only deployment cost matters |
| PEFT + quantization | edge deployment or many domain adapters |
| Avoid aggressive quantization | safety-critical cases where quality loss is unacceptable |

For support automation, a safe process is:

```text
SFT/LoRA → behaviour evaluation → quantization → repeat evaluation → staged deployment
```

# Part 17 — Deployment tools map

| Tool | Where it fits |
|---|---|
| Hugging Face Transformers | model loading, tokenization, SFT |
| PEFT | LoRA/adapters |
| Accelerate | easier device placement and distributed training |
| bitsandbytes | 8-bit/4-bit model loading on compatible GPUs |
| ONNX Runtime | optimized ONNX inference |
| Intel Neural Compressor | compression/quantization workflows |
| OpenVINO | optimized deployment on Intel hardware |

# Part 18 — Safe deployment checklist

Before deploying a fine-tuned or quantized model:

1. Review training data quality.
2. Test safety-critical cases.
3. Compare base vs adapted behaviour.
4. Compare adapted vs quantized behaviour.
5. Check hallucination risk.
6. Check latency.
7. Keep rollback option.
8. Route high-risk actions to humans.

In [ ]:
deployment_checklist = [
    {"check": "Training data reviewed", "required": True},
    {"check": "Safety cases evaluated", "required": True},
    {"check": "Refund/date hallucination tested", "required": True},
    {"check": "Base vs adapted comparison done", "required": True},
    {"check": "Quantized model evaluated", "required": True},
    {"check": "Human escalation path exists", "required": True},
    {"check": "Monitoring and rollback plan exists", "required": True},
]

print(json.dumps(deployment_checklist, indent=2))

[
  {
    "check": "Training data reviewed",
    "required": true
  },
  {
    "check": "Safety cases evaluated",
    "required": true
  },
  {
    "check": "Refund/date hallucination tested",
    "required": true
  },
  {
    "check": "Base vs adapted comparison done",
    "required": true
  },
  {
    "check": "Quantized model evaluated",
    "required": true
  },
  {
    "check": "Human escalation path exists",
    "required": true
  },
  {
    "check": "Monitoring and rollback plan exists",
    "required": true
  }
]


# What this module should make students understand

By this point, students should be able to explain:

1. Why fine-tuning is different from prompting.
2. Why SFT needs carefully designed examples.
3. Why LoRA/PEFT is useful when full fine-tuning is too expensive.
4. Why evaluation must include safety and generalization.
5. Why quantization helps deployment but can reduce quality.
6. Why INT8 is often safer than INT4 for sensitive tasks.
7. Why deployment decisions require accuracy, size, latency, and safety evidence.
8. How this connects to Module 25 feedback-driven learning.

# Part 19 — Final student challenge

Students must submit a mini deployment recommendation.

Include:

1. SFT dataset sample.
2. Full model trainable parameter count.
3. Adapter trainable parameter count.
4. Behaviour evaluation results.
5. FP32 vs INT8 size.
6. FP32 vs INT8 latency.
7. Final recommendation:
   - prompting only?
   - SFT?
   - LoRA/adapter?
   - quantization?
   - LoRA + quantization?
8. Safety note before deployment.

# Closing summary

You completed a stable Module 24 workflow:

```text
SFT data
  → adaptation idea
  → full model training
  → adapter-style training
  → evaluation
  → quantization
  → deployment decision
```

The next module, Module 25, continues naturally:

```text
What if we do not only have correct examples,
but also human preferences and feedback?
```

That leads to:

```text
RLHF → reward models → DPO → PPO
```